In [ ]:
import os
import zipfile
import shutil

# 1. Configurar la ruta para que la API busque el kaggle.json en la raíz del proyecto (un nivel arriba de /src)
ruta_raiz = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.environ['KAGGLE_CONFIG_DIR'] = ruta_raiz

# Definir rutas de destino
ruta_kaggle = os.path.join(ruta_raiz, "kaggle.json")
ruta_dataset_raw = os.path.join(ruta_raiz, "dataset_raw")

# Verificar existencia de credenciales
if not os.path.exists(ruta_kaggle):
    raise FileNotFoundError(
        "No se encontró kaggle.json en la raíz del proyecto. Descargalo desde Kaggle y colocá ese archivo con ese nombre exacto."
    )

# Si la carpeta dataset_raw ya contiene archivos, preguntar si borrar su contenido
if os.path.exists(ruta_dataset_raw) and any(os.scandir(ruta_dataset_raw)):
    respuesta = input(f"La carpeta '{ruta_dataset_raw}' ya contiene archivos. Eliminar su contenido y descargar de nuevo? [y/N]: ")
    if respuesta.strip().lower() in ("y", "yes"):
        shutil.rmtree(ruta_dataset_raw)
        os.makedirs(ruta_dataset_raw, exist_ok=True)
        print("Contenido de dataset_raw eliminado. Procediendo a descargar de nuevo...")
    else:
        print("Conservando archivos existentes en dataset_raw. La descarga podría reutilizar archivos ya presentes.")

print(f"Archivo de credenciales: {ruta_kaggle}")

In [ ]:
ruta_zip = os.path.join(ruta_dataset_raw, "cifake-real-and-ai-generated-synthetic-images.zip")

print("Iniciando descarga del dataset desde Kaggle...")

# Asegurar que exista la carpeta contenedora
os.makedirs(ruta_dataset_raw, exist_ok=True)
try:
    # 2. Descargar usando la API oficial
    # Usamos un script de python en lugar de ! para mantener compatibilidad pura de entornos
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files('birdy654/cifake-real-and-ai-generated-synthetic-images', path=ruta_dataset_raw, quiet=False)

    # 3. Descomprimir el archivo zip descargado
    if os.path.exists(ruta_zip):
        print("\nDescomprimiendo archivos en dataset_raw/...")
        with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
            zip_ref.extractall(ruta_dataset_raw)

        # Remover el archivo comprimido para optimizar espacio en disco local
        os.remove(ruta_zip)
        print("✅ ¡Dataset descargado, descomprimido y listo de manera automática!")
    else:
        # A veces la API descarga directamente las carpetas o un zip con otro nombre interno según la versión
        print("✅ Descarga completada. Si no ves el archivo ZIP, revisá la estructura de 'dataset_raw/'")

except Exception as e:
    print(f"❌ Error en la descarga: {e}")
    print("Asegurate de haber colocado el archivo 'config.json' correctamente en la raíz del proyecto.")